In [0]:
%sql
-- ============================================================
-- TABELA: GERAÇÃO POR USINA - BASE HORÁRIA
-- FONTE: ONS - Dados Abertos
-- ============================================================

CREATE TABLE IF NOT EXISTS mba.raw.ons_geracao_usina_horaria
(
    -- ========================================================
    -- DADOS DE NEGÓCIO
    -- ========================================================

    din_instante TIMESTAMP NOT NULL
        COMMENT 'Data e hora de referência da geração da usina. Formato original: YYYY-MM-DD HH:MM:SS.',

    id_subsistema STRING NOT NULL
        COMMENT 'Identificador do subsistema onde está localizada a usina. Código de até 3 posições.',

    nom_subsistema STRING NOT NULL
        COMMENT 'Nome do subsistema onde está localizada a usina.',

    id_estado STRING NOT NULL
        COMMENT 'Identificador do estado onde está localizada a usina. Código de 2 posições.',

    nom_estado STRING NOT NULL
        COMMENT 'Nome do estado onde está localizada a usina.',

    cod_modalidadeoperacao STRING
        COMMENT 'Código da modalidade de operação da usina. Campo que pode possuir valor nulo.',

    nom_tipousina STRING NOT NULL
        COMMENT 'Tipo da usina.',

    nom_tipocombustivel STRING NOT NULL
        COMMENT 'Tipo de combustível utilizado pela usina.',

    nom_usina STRING NOT NULL
        COMMENT 'Nome da usina. Até 60 posições.',

    id_ons STRING
        COMMENT 'Identificador ONS da usina. Até 32 posições.',

    ceg STRING
        COMMENT 'Código Único do Empreendimento de Geração (CEG), estabelecido pela ANEEL. Campo que pode possuir valor nulo.',

    val_geracao DOUBLE
        COMMENT 'Geração de energia da usina em MWmed. Permite valor zero e não permite valor negativo.',


    -- ========================================================
    -- METADADOS DE INGESTÃO
    -- ========================================================

    NmArquivoCarga STRING
        COMMENT 'Nome do arquivo de origem utilizado na carga.',

    DatCarga TIMESTAMP
        COMMENT 'Data e hora em que o registro foi carregado na Delta Table.'

)
USING DELTA

COMMENT 'Geração verificada de usinas, conjuntos de usinas e grupos de pequenas usinas em base horária, conforme dicionário de dados do ONS.'
;

In [0]:
from pyspark.sql import functions as F

# ============================================================
# CONFIGURAÇÕES
# ============================================================

CAMINHO_ORIGEM = "/Volumes/mba/stage/dados_bruto/usinas/GERACAO_USINA"

TABELA_DESTINO = "mba.raw.ons_geracao_usina_horaria"


# ============================================================
# 1. LISTA OS ARQUIVOS PARQUET DA ORIGEM
# ============================================================

arquivos = [
    arquivo
    for arquivo in dbutils.fs.ls(CAMINHO_ORIGEM)
    if arquivo.path.lower().endswith(".parquet")
]

print(f"Quantidade de arquivos encontrados na origem: {len(arquivos)}")


# ============================================================
# 2. IDENTIFICA OS ARQUIVOS JÁ CARREGADOS
# ============================================================

if spark.catalog.tableExists(TABELA_DESTINO):

    arquivos_carregados = {
        row["NmArquivoCarga"]
        for row in (
            spark.table(TABELA_DESTINO)
            .select("NmArquivoCarga")
            .where(F.col("NmArquivoCarga").isNotNull())
            .distinct()
            .collect()
        )
    }

else:

    arquivos_carregados = set()


print(f"Quantidade de arquivos já carregados: {len(arquivos_carregados)}")


# ============================================================
# 3. FILTRA SOMENTE OS ARQUIVOS NOVOS
# ============================================================

arquivos_novos = [
    arquivo
    for arquivo in arquivos
    if arquivo.name not in arquivos_carregados
]

print(f"Quantidade de arquivos novos: {len(arquivos_novos)}")


# ============================================================
# 4. LEITURA INDIVIDUAL DOS ARQUIVOS
# ============================================================

dfs = []

for arquivo in arquivos_novos:

    #print(f"Lendo: {arquivo.name}")

    # --------------------------------------------------------
    # Lê somente um arquivo
    # --------------------------------------------------------

    df_temp = spark.read.parquet(arquivo.path)

    # --------------------------------------------------------
    # Metadados de carga
    # --------------------------------------------------------

    df_temp = (
        df_temp
        .withColumn(
            "NmArquivoCarga",
            F.lit(arquivo.name)
        )
        .withColumn(
            "DatCarga",
            F.current_timestamp()
        )
    )

    # --------------------------------------------------------
    # Padroniza VAL_GERACAO
    #
    # Independente de o Parquet original possuir:
    # STRING, DOUBLE etc.
    #
    # Primeiro converte para STRING.
    # Depois converte para DOUBLE.
    #
    # '' -> NULL
    # valor inválido -> NULL
    # --------------------------------------------------------

    df_temp = df_temp.withColumn(
        "val_geracao",
        F.expr("""
            try_cast(
                nullif(trim(cast(val_geracao AS STRING)), '')
                AS DOUBLE
            )
        """)
    )

    dfs.append(df_temp)


# ============================================================
# 5. UNE OS DATAFRAMES
# ============================================================

if len(dfs) > 0:

    df_novos = dfs[0]

    for df_temp in dfs[1:]:
        df_novos = df_novos.unionByName(
            df_temp,
            allowMissingColumns=True
        )

else:

    df_novos = None

# ============================================================
# 7. GRAVA NA DELTA TABLE
# ============================================================

if df_novos is not None:

    (
        df_novos
        .select(
            "din_instante",
            "id_subsistema",
            "nom_subsistema",
            "id_estado",
            "nom_estado",
            "cod_modalidadeoperacao",
            "nom_tipousina",
            "nom_tipocombustivel",
            "nom_usina",
            "id_ons",
            "ceg",
            "val_geracao",
            "NmArquivoCarga",
            "DatCarga"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABELA_DESTINO)
    )

    print("============================================")
    print("Carga realizada com sucesso!")
    print("============================================")

else:

    print("============================================")
    print("Nenhum arquivo novo para carregar.")
    print("============================================")

In [0]:
dbutils.notebook.exit("OK")

In [0]:
%sql
select * from mba.raw.ons_geracao_usina_horaria